# Notebook 03 — Final Comparison: Pipeline A versus Pipeline B

**Project:** Reproducible comparison of EEG preprocessing pipelines on OpenNeuro `ds004504`.

---

## Why this is a separate notebook

Notebooks 01 and 02 each *produce* results. This one only *consumes* them. Keeping the
final analysis separate means:

- the two preprocessing notebooks stay independently reproducible;
- the comparison can be re-inspected, re-run and audited without re-processing 5 GB of EEG;
- no analysis choice made here can leak backwards into how either pipeline was run.

This notebook loads only saved result files. It does not touch the EEG data.

## The question being answered

> Does an alternative, deterministic, ICA-free preprocessing strategy deliver comparable
> EEG signal quality and comparable diagnostic classification performance to the dataset
> authors' published ASR + ICA + ICLabel pipeline, at lower computational cost?

## The standard of evidence

The goal is **not** to make Pipeline B win. All six of these are acceptable conclusions:

1. B is statistically better.
2. B is statistically comparable but computationally cheaper.
3. B is cheaper but scientifically worse.
4. B is better but more expensive.
5. No meaningful difference.
6. Evidence is insufficient.

We report whichever the measurements support.

> **Prerequisites:** run Notebooks 01 and 02 first.

## 1. Setup

### 1.1 Mount Drive and load the shared module

In [ ]:
# Google Drive is OPTIONAL. It is worth using because the cache and results survive a
# Colab session timeout -- without it, a disconnect means reprocessing from scratch.
# Set USE_DRIVE = False to run entirely on the ephemeral Colab disk (or locally).
# The DATASET itself is always downloaded at runtime and never needs to be in Drive.

USE_DRIVE = True          # <-- set False to skip Drive entirely

IN_COLAB = False
DRIVE_OK = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB and USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OK = True
        print("Drive mounted: results and cache will persist across sessions.")
    except Exception as exc:
        print(f"Drive mount failed ({exc.__class__.__name__}): {exc}")
        print("Continuing on the local Colab disk. Results will be LOST on disconnect "
              "unless you download them before the session ends.")
elif IN_COLAB:
    print("USE_DRIVE = False. Working on the local Colab disk; "
          "download your results before the session ends.")
else:
    print("Not running in Colab. Using local directories.")

In [ ]:
# `ds004504_common.py` holds every pipeline, feature, cross-validation and statistics
# function. Both preprocessing notebooks import the SAME module, which is how the
# "fair comparison" requirement is enforced structurally rather than by convention.
#
# This cell looks for the module in the usual places and, if it cannot find it,
# opens Colab's file picker so you can upload it directly -- no Drive required.
import sys
from pathlib import Path

CANDIDATE_DIRS = [
    "/content/drive/MyDrive/ds004504_experiment",   # Drive, if mounted
    "/content",                                      # Colab working dir
    ".",                                             # local / cwd
]

def _locate_module():
    for d in CANDIDATE_DIRS:
        if (Path(d) / "ds004504_common.py").exists():
            return d
    return None

MODULE_DIR = _locate_module()

if MODULE_DIR is None and IN_COLAB:
    print("ds004504_common.py not found. Please upload it now.")
    try:
        from google.colab import files
        files.upload()
        MODULE_DIR = _locate_module()
    except Exception as exc:
        print(f"Upload failed: {exc!r}")

if MODULE_DIR is None:
    raise FileNotFoundError(
        "ds004504_common.py could not be found in any of: "
        + ", ".join(CANDIDATE_DIRS)
        + ". Upload it next to this notebook (or into Drive) and re-run."
    )

if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

import ds004504_common as ds
print(f"Loaded ds004504_common v{ds.MODULE_VERSION} from {MODULE_DIR}")

### 1.2 Imports and plotting defaults

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

# Publication-quality defaults.
plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 300, "font.size": 10,
    "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
    "axes.spines.right": False, "figure.facecolor": "white",
    "axes.titlesize": 11, "axes.labelsize": 10, "legend.frameon": False,
})

COLOR_A, COLOR_B = "#4c72b0", "#dd8452"   # consistent across every figure
print("Imports ready.")

### 1.3 Configuration

We reload the configuration that Notebooks 01 and 02 actually used, rather than re-declaring it. That guarantees this analysis describes the run that happened.

In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/ds004504_experiment" if IN_COLAB else "./ds004504_experiment"

cfg = ds.Config(output_path=OUTPUT_PATH)
cfg.make_dirs()

cfg_path = cfg.results_dir / "config.json"
if not cfg_path.exists():
    raise FileNotFoundError(
        f"No config.json at {cfg_path}. Run Notebooks 01 and 02 first."
    )

with open(cfg_path) as fh:
    saved_cfg = json.load(fh)

# Rebuild the exact configuration used upstream, so seeds and CV settings match.
for key, value in saved_cfg.items():
    if key.startswith("_"):
        continue
    if hasattr(cfg, key) and key not in ("dataset_path", "output_path"):
        try:
            setattr(cfg, key, value)
        except Exception:
            pass

print(f"Run mode        : {saved_cfg.get('_mode')}")
print(f"Max subjects    : {cfg.max_subjects}")
print(f"Random seed     : {cfg.random_seed}")
print(f"Filter          : {cfg.l_freq}-{cfg.h_freq} Hz")
print(f"Reference       : {cfg.reference}")
print(f"Epoch length    : {cfg.epoch_length_s} s")
print(f"CV              : {cfg.cv_n_splits}-fold x {cfg.cv_n_repeats} repeats")
print(f"Bootstrap draws : {cfg.n_bootstrap}")

### 1.4 Load every upstream result

Anything missing is reported explicitly. Downstream cells then state *"not computed because..."* rather than silently omitting a row.

In [ ]:
def try_load(name):
    """Load a result file, returning None (with a note) if the producer never ran."""
    try:
        return ds.load_results(cfg, name)
    except FileNotFoundError:
        MISSING.append(name)
        return None

MISSING = []
R = {}
for name in [
    "environment", "config",
    "classification_results_A", "classification_results_B",
    "classification_results_A_rf", "classification_results_B_rf",
    "fold_results_A", "fold_results_B",
    "oof_predictions_A", "oof_predictions_B",
    "runtime_results_A", "runtime_results_B",
    "signal_quality_results_A", "signal_quality_results_B",
    "scalability_A", "scalability_B",
    "storage_A", "storage_B",
    "exclusion_report_A", "exclusion_report_B",
    "loso_results_A", "loso_results_B",
    "bad_channel_stats_B", "novelty_assessment", "hypotheses",
    "literature_comparison_A", "pipeline_a_uncertainties",
    "subject_metadata", "dataset_verification",
]:
    R[name] = try_load(name)

print(f"Loaded {sum(v is not None for v in R.values())}/{len(R)} result files.")
if MISSING:
    print("\nMISSING (analyses depending on these will be reported as not computed):")
    for m in MISSING:
        print("  -", m)
else:
    print("All expected result files present.")

if R["classification_results_A"] is None or R["classification_results_B"] is None:
    raise FileNotFoundError(
        "Classification results for both pipelines are required. "
        "Run Notebooks 01 and 02 to completion first."
    )

### 1.5 Establish the paired subject set

Every paired statistic below is restricted to subjects that **both** pipelines processed successfully. Comparing different subject sets would confound the pipeline effect with sampling.

In [ ]:
oof_A = R["oof_predictions_A"]
oof_B = R["oof_predictions_B"]

subs_A = set(oof_A["participant_id"].unique())
subs_B = set(oof_B["participant_id"].unique())
PAIRED = sorted(subs_A & subs_B)

print(f"Subjects with Pipeline A results : {len(subs_A)}")
print(f"Subjects with Pipeline B results : {len(subs_B)}")
print(f"Paired (used for all comparisons): {len(PAIRED)}")

only_A, only_B = subs_A - subs_B, subs_B - subs_A
if only_A: print(f"\nExcluded (A only): {sorted(only_A)}")
if only_B: print(f"Excluded (B only): {sorted(only_B)}")

TASKS = sorted(set(oof_A["task"]) & set(oof_B["task"]))
print(f"\nTasks available in both: {TASKS}")

TASK_CLASSES = {"AD_vs_CN": ["A", "C"], "AD_vs_FTD": ["A", "F"],
                "AD_vs_FTD_vs_CN": ["A", "C", "F"]}

if len(PAIRED) < 10:
    print(f"\nWARNING: only {len(PAIRED)} paired subjects. Statistical comparisons will "
          "be severely underpowered. Treat all inference below as provisional and "
          "re-run in FULL mode before drawing conclusions.")

---
## 2. Classification performance

### 2.1 Side-by-side summary

In [ ]:
def summary_row(res, pipeline, task):
    s = res.get(task, {}).get("summary", {})
    if not s:
        return None
    row = {"pipeline": pipeline, "task": task}
    for m in ["balanced_accuracy", "accuracy", "f1_macro", "f1_weighted",
              "precision_macro", "recall_macro", "roc_auc", "pr_auc",
              "sensitivity", "specificity", "roc_auc_ovr_macro"]:
        if f"{m}_mean" in s:
            row[m] = round(s[f"{m}_mean"], 4)
            row[f"{m}_sd"] = round(s.get(f"{m}_std", np.nan), 4)
    return row

rows = []
for task in TASKS:
    for pipeline, res in [("A", R["classification_results_A"]),
                          ("B", R["classification_results_B"])]:
        r = summary_row(res, pipeline, task)
        if r:
            rows.append(r)

perf = pd.DataFrame(rows)
ds.save_results(cfg, "comparison_performance_summary", perf)

for task in TASKS:
    sub = perf[perf["task"] == task]
    if sub.empty:
        print(f"\n{task}: not computed.")
        continue
    print(f"\n{'=' * 74}\n{task}\n{'=' * 74}")
    cols = [c for c in ["pipeline", "balanced_accuracy", "balanced_accuracy_sd",
                        "accuracy", "f1_macro", "roc_auc", "sensitivity", "specificity"]
            if c in sub.columns]
    display(sub[cols].set_index("pipeline"))

### 2.2 Statistical comparison

Four complementary tests, in decreasing order of authority:

| Test | Unit | Status | Why |
|---|---|---|---|
| **Paired bootstrap over subjects** | Subject | **PRIMARY** | Subjects are the genuine independent sampling units; resampling them gives a valid interval for the difference |
| Exact McNemar | Subject | Supporting | Directly asks whether the subjects the pipelines disagree on split lopsidedly |
| Nadeau–Bengio corrected *t* | Fold | Supporting | Inflates the variance to account for overlapping training sets |
| Wilcoxon signed-rank | Fold | Exploratory only | Common in the literature but **anti-conservative** here, because folds are not independent |

The **95% confidence interval for the difference is the headline result**, not the
*p*-value. If that interval contains zero, the data do not support a difference — which
for a hypothesis of *equivalence* is an informative outcome, not a null result.

In [ ]:
stat_rows = []
stat_detail = {}

for task in TASKS:
    a = oof_A[(oof_A["task"] == task) & (oof_A["participant_id"].isin(PAIRED))]
    b = oof_B[(oof_B["task"] == task) & (oof_B["participant_id"].isin(PAIRED))]
    if a.empty or b.empty:
        print(f"{task}: not computed (missing predictions).")
        continue

    classes = sorted(TASK_CLASSES.get(task, sorted(a["true_class"].unique())))
    detail = {}

    for metric in ["balanced_accuracy", "accuracy", "f1_macro"] + (
            ["roc_auc"] if len(classes) == 2 else []):
        bs = ds.paired_bootstrap_subject_level(
            a, b, metric=metric, classes=classes,
            n_boot=cfg.n_bootstrap, seed=cfg.random_seed)
        if "error" in bs:
            print(f"{task}/{metric}: {bs['error']}")
            continue
        detail[metric] = bs
        crosses_zero = bs["ci95_low"] <= 0 <= bs["ci95_high"]
        stat_rows.append({
            "task": task, "metric": metric,
            "pipeline_A": round(bs["pipeline_A"], 4),
            "pipeline_B": round(bs["pipeline_B"], 4),
            "delta_B_minus_A": round(bs["observed_delta_B_minus_A"], 4),
            "ci95_low": round(bs["ci95_low"], 4),
            "ci95_high": round(bs["ci95_high"], 4),
            "standardised_effect": round(bs["standardised_effect"], 3),
            "p_value": round(bs["p_value_two_sided"], 4),
            "n_subjects": bs["n_subjects"],
            "ci_includes_zero": crosses_zero,
            "verdict": ("no evidence of difference" if crosses_zero
                        else ("B higher" if bs["observed_delta_B_minus_A"] > 0 else "A higher")),
        })

    detail["mcnemar"] = ds.mcnemar_subject_level(a, b)
    stat_detail[task] = detail

stats_table = pd.DataFrame(stat_rows)
ds.save_results(cfg, "comparison_statistics", stats_table)
ds.save_results(cfg, "comparison_statistics_detail", stat_detail)

if not stats_table.empty:
    display(stats_table[["task", "metric", "pipeline_A", "pipeline_B",
                         "delta_B_minus_A", "ci95_low", "ci95_high",
                         "p_value", "verdict"]])
else:
    print("No statistical comparisons could be computed.")

#### 2.2.1 McNemar — where exactly do the pipelines disagree?

In [ ]:
mc_rows = []
for task, detail in stat_detail.items():
    mc = detail.get("mcnemar", {})
    if "error" in mc:
        continue
    mc_rows.append({
        "task": task,
        "n_subjects": mc.get("n_subjects"),
        "both_correct": mc.get("both_correct"),
        "both_wrong": mc.get("both_wrong"),
        "only_A_correct": mc.get("only_A_correct"),
        "only_B_correct": mc.get("only_B_correct"),
        "n_discordant": mc.get("n_discordant"),
        "p_value": round(mc["p_value"], 4) if "p_value" in mc else None,
    })

mcnemar_table = pd.DataFrame(mc_rows)
ds.save_results(cfg, "comparison_mcnemar", mcnemar_table)
if not mcnemar_table.empty:
    display(mcnemar_table)
    print("\nA small discordant count means the pipelines classify almost the same "
          "people the same way -- strong evidence of practical equivalence, regardless "
          "of the p-value. A large but balanced discordant count means they disagree "
          "often without either being better.")
else:
    print("McNemar: not computed.")

#### 2.2.2 Fold-level tests (supporting and exploratory)

In [ ]:
fa, fb = R["fold_results_A"], R["fold_results_B"]

if fa is None or fb is None:
    print("Fold-level tests: NOT COMPUTED (fold result files missing).")
else:
    rows = []
    for task in TASKS:
        sa = fa[fa["task"] == task]
        sb = fb[fb["task"] == task]
        if sa.empty or sb.empty:
            continue

        n_test = int(sa["n_test_subjects"].mean()) if "n_test_subjects" in sa else None
        n_train = (len(PAIRED) - n_test) if n_test else None

        nb_t = ds.corrected_resampled_ttest(sa, sb, "balanced_accuracy",
                                            n_train=n_train, n_test=n_test)
        wil = ds.wilcoxon_fold_level(sa, sb, "balanced_accuracy")

        rows.append({
            "task": task,
            "n_folds": nb_t.get("n_pairs"),
            "mean_delta": round(nb_t["mean_delta_B_minus_A"], 4) if "mean_delta_B_minus_A" in nb_t else None,
            "NB_corrected_t": round(nb_t["t_statistic"], 3) if "t_statistic" in nb_t else None,
            "NB_p_value": round(nb_t["p_value"], 4) if "p_value" in nb_t else None,
            "NB_inflation_factor": round(nb_t["inflation_factor"], 2) if "inflation_factor" in nb_t else None,
            "Wilcoxon_p_ANTICONSERVATIVE": round(wil["p_value"], 4) if "p_value" in wil else None,
        })

    fold_tests = pd.DataFrame(rows)
    ds.save_results(cfg, "comparison_fold_tests", fold_tests)
    if not fold_tests.empty:
        display(fold_tests)
        print("\nThe inflation factor shows how much the Nadeau-Bengio correction widens "
              "the standard error relative to naively treating folds as independent. "
              "Where the Wilcoxon p-value is much smaller than the corrected one, that "
              "gap is precisely the over-optimism the correction exists to remove.")
    else:
        print("Fold-level tests: no matching tasks.")

### 2.3 Performance comparison figure

In [ ]:
if not perf.empty:
    metrics = [m for m in ["balanced_accuracy", "f1_macro", "roc_auc"]
               if m in perf.columns and perf[m].notna().any()]
    fig, axes = plt.subplots(1, len(metrics), figsize=(5.2 * len(metrics), 4.6))
    axes = np.atleast_1d(axes)

    for ax, metric in zip(axes, metrics):
        tasks_here = [t for t in TASKS if perf[(perf["task"] == t)][metric].notna().any()]
        x = np.arange(len(tasks_here)); w = 0.36
        for off, pipeline, colour, lab in [(-w/2, "A", COLOR_A, "Pipeline A (authors')"),
                                           (+w/2, "B", COLOR_B, "Pipeline B (alternative)")]:
            vals, errs = [], []
            for t in tasks_here:
                r = perf[(perf["task"] == t) & (perf["pipeline"] == pipeline)]
                vals.append(r[metric].iloc[0] if not r.empty else np.nan)
                sd_col = f"{metric}_sd"
                errs.append(r[sd_col].iloc[0] if (not r.empty and sd_col in r) else 0)
            ax.bar(x + off, vals, w, yerr=errs, capsize=3, label=lab,
                   color=colour, edgecolor="black", linewidth=0.6)

        ax.axhline(0.5, color="red", ls=":", lw=1, label="Chance (binary)")
        ax.set_xticks(x, [t.replace("_", " ") for t in tasks_here], rotation=15, ha="right")
        ax.set_ylabel(metric.replace("_", " ").title())
        ax.set_ylim(0, 1.05)
        ax.set_title(metric.replace("_", " ").title())

    axes[0].legend(loc="lower right", fontsize=8)
    plt.suptitle("Classification performance by preprocessing pipeline\n"
                 "(subject-level; error bars = SD across CV folds)", y=1.04)
    plt.tight_layout()
    plt.savefig(cfg.figures_dir / "fig_performance_comparison.png", bbox_inches="tight")
    plt.show()

### 2.4 Effect sizes with confidence intervals

The most important figure in the notebook. Each interval that straddles zero is evidence of equivalence on that metric.

In [ ]:
if not stats_table.empty:
    st = stats_table.copy()
    st["label"] = st["task"].str.replace("_", " ") + "\n" + st["metric"].str.replace("_", " ")
    st = st.iloc[::-1].reset_index(drop=True)   # first task at the top

    fig, ax = plt.subplots(figsize=(9, 0.52 * len(st) + 2.2))
    y = np.arange(len(st))
    colours = ["#888888" if z else (COLOR_B if d > 0 else COLOR_A)
               for z, d in zip(st["ci_includes_zero"], st["delta_B_minus_A"])]

    # matplotlib's `ecolor` takes a single colour, not a per-point list, so each
    # interval is drawn individually to colour it by its own verdict.
    for yi, (lo, hi, d, colour) in enumerate(
            zip(st["ci95_low"], st["ci95_high"], st["delta_B_minus_A"], colours)):
        ax.errorbar(d, yi, xerr=[[d - lo], [hi - d]], fmt="none",
                    ecolor=colour, elinewidth=2.2, capsize=4)
    ax.scatter(st["delta_B_minus_A"], y, c=colours, s=70, zorder=3,
               edgecolor="black", linewidth=0.6)
    ax.axvline(0, color="black", ls="--", lw=1.2)

    ax.set_yticks(y, st["label"], fontsize=8)
    ax.set_xlabel("Difference in metric (Pipeline B − Pipeline A)\n"
                  "← Pipeline A better    |    Pipeline B better →")
    ax.set_title("Paired bootstrap: difference between pipelines with 95% CI\n"
                 f"({len(PAIRED)} paired subjects, {cfg.n_bootstrap} bootstrap resamples)")
    ax.legend(handles=[
        Patch(facecolor="#888888", label="CI includes 0 — no evidence of difference"),
        Patch(facecolor=COLOR_B, label="Pipeline B higher"),
        Patch(facecolor=COLOR_A, label="Pipeline A higher"),
    ], loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(cfg.figures_dir / "fig_effect_sizes.png", bbox_inches="tight")
    plt.show()
else:
    print("Effect-size figure: NOT COMPUTED (no statistics available).")

### 2.5 Robustness — does the secondary classifier agree?

If the two classifiers disagree about which pipeline is better, no conclusion about preprocessing is safe.

In [ ]:
rf_A, rf_B = R["classification_results_A_rf"], R["classification_results_B_rf"]
if rf_A is None or rf_B is None:
    print("Secondary-classifier check: NOT COMPUTED (results missing).")
else:
    rows = []
    for task in TASKS:
        for label, ra, rb in [("logreg (primary)", R["classification_results_A"],
                               R["classification_results_B"]),
                              ("random forest", rf_A, rf_B)]:
            sa = ra.get(task, {}).get("summary", {})
            sb = rb.get(task, {}).get("summary", {})
            if not sa or not sb:
                continue
            rows.append({
                "task": task, "classifier": label,
                "A": round(sa["balanced_accuracy_mean"], 4),
                "B": round(sb["balanced_accuracy_mean"], 4),
                "delta_B_minus_A": round(sb["balanced_accuracy_mean"]
                                         - sa["balanced_accuracy_mean"], 4),
            })
    robust = pd.DataFrame(rows)
    ds.save_results(cfg, "comparison_classifier_robustness", robust)
    if not robust.empty:
        display(robust.pivot_table(index="task", columns="classifier",
                                   values="delta_B_minus_A").round(4))
        signs = robust.groupby("task")["delta_B_minus_A"].apply(
            lambda s: len(set(np.sign(s))) == 1)
        print("\nDirection of the difference agrees across classifiers:")
        for task, agrees in signs.items():
            print(f"  {task:<20} {'yes' if agrees else 'NO - conclusion is classifier-dependent'}")

### 2.6 Robustness — does LOSO agree with repeated group k-fold?

In [ ]:
la, lb = R["loso_results_A"], R["loso_results_B"]
if la is None or lb is None:
    print("LOSO comparison: NOT COMPUTED (LOSO results missing).")
else:
    rows = []
    for task in TASKS:
        pa = la.get(task, {}).get("pooled_subject_level", {})
        pb = lb.get(task, {}).get("pooled_subject_level", {})
        if not pa or not pb:
            continue
        rows.append({
            "task": task,
            "A_loso": round(pa["balanced_accuracy"], 4),
            "B_loso": round(pb["balanced_accuracy"], 4),
            "delta_loso": round(pb["balanced_accuracy"] - pa["balanced_accuracy"], 4),
        })
    loso_cmp = pd.DataFrame(rows)
    ds.save_results(cfg, "comparison_loso", loso_cmp)
    if not loso_cmp.empty:
        merged = loso_cmp.merge(
            stats_table[stats_table["metric"] == "balanced_accuracy"][
                ["task", "delta_B_minus_A"]].rename(
                    columns={"delta_B_minus_A": "delta_sgkf"}), on="task", how="left")
        display(merged)
        print("\nIf delta_loso and delta_sgkf have the same sign and similar magnitude, "
              "the conclusion is not an artefact of the validation scheme.")

---
## 3. Signal-quality comparison

Classification accuracy alone cannot tell us whether a pipeline produced *good EEG* — a
pipeline could destroy signal and still classify well if what survives happens to
correlate with diagnosis. We therefore compare the cleaned signals directly.

In [ ]:
qa, qb = R["signal_quality_results_A"], R["signal_quality_results_B"]

if qa is None or qb is None:
    print("Signal-quality comparison: NOT COMPUTED (quality files missing).")
    quality_cmp = None
else:
    qa = qa[qa["participant_id"].isin(PAIRED)].set_index("participant_id").sort_index()
    qb = qb[qb["participant_id"].isin(PAIRED)].set_index("participant_id").sort_index()
    common = qa.index.intersection(qb.index)
    qa, qb = qa.loc[common], qb.loc[common]

    METRICS = [
        ("after_rms_v", "RMS amplitude (V)", "time domain"),
        ("after_variance_v2", "Variance (V²)", "time domain"),
        ("after_kurtosis_mean", "Mean kurtosis", "time domain"),
        ("after_frac_samples_robust_z_gt5", "Fraction |z|>5 samples", "artifact proxy"),
        ("after_frac_samples_robust_z_gt10", "Fraction |z|>10 samples", "artifact proxy"),
        ("after_relpow_delta", "Relative delta power", "frequency"),
        ("after_relpow_theta", "Relative theta power", "frequency"),
        ("after_relpow_alpha", "Relative alpha power", "frequency"),
        ("after_relpow_beta", "Relative beta power", "frequency"),
        ("after_spectral_entropy_mean", "Spectral entropy", "frequency"),
        ("after_sef95_mean_hz", "95% spectral edge (Hz)", "frequency"),
    ]

    rows = []
    for col, label, domain in METRICS:
        if col not in qa.columns or col not in qb.columns:
            continue
        va, vb = qa[col].to_numpy(float), qb[col].to_numpy(float)
        ok = np.isfinite(va) & np.isfinite(vb)
        va, vb = va[ok], vb[ok]
        if len(va) < 3:
            continue
        d = vb - va
        # Paired Wilcoxon is valid here: each subject contributes ONE pair, and
        # subjects are independent (unlike CV folds).
        try:
            from scipy.stats import wilcoxon
            stat, p = wilcoxon(va, vb) if not np.allclose(va, vb) else (np.nan, 1.0)
        except Exception:
            stat, p = np.nan, np.nan
        rows.append({
            "domain": domain, "metric": label,
            "pipeline_A_mean": float(np.mean(va)),
            "pipeline_B_mean": float(np.mean(vb)),
            "mean_diff_B_minus_A": float(np.mean(d)),
            "rel_diff_pct": float(np.mean(d) / np.mean(va) * 100) if np.mean(va) else np.nan,
            "wilcoxon_p": float(p),
            "n_subjects": int(len(va)),
        })

    quality_cmp = pd.DataFrame(rows)
    ds.save_results(cfg, "comparison_signal_quality", quality_cmp)
    if not quality_cmp.empty:
        display(quality_cmp.round(5))
        print("\nThis Wilcoxon test IS appropriate: each subject contributes one paired "
              "observation and subjects are independent -- unlike the fold-level test in "
              "section 2.2.2.")

### 3.1 Signal-quality figure

In [ ]:
if quality_cmp is not None and not quality_cmp.empty:
    band_rows = quality_cmp[quality_cmp["metric"].str.contains("Relative")]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))

    # (a) relative band power
    if not band_rows.empty:
        x = np.arange(len(band_rows)); w = 0.36
        axes[0].bar(x - w/2, band_rows["pipeline_A_mean"], w, label="Pipeline A",
                    color=COLOR_A, edgecolor="black", linewidth=0.6)
        axes[0].bar(x + w/2, band_rows["pipeline_B_mean"], w, label="Pipeline B",
                    color=COLOR_B, edgecolor="black", linewidth=0.6)
        axes[0].set_xticks(x, [m.replace("Relative ", "").replace(" power", "")
                               for m in band_rows["metric"]])
        axes[0].set_ylabel("Relative band power (fraction of total)")
        axes[0].set_xlabel("Frequency band")
        axes[0].set_title("(a) Spectral content")
        axes[0].legend(fontsize=8)

    # (b) residual artifact indicator, per subject
    col = "after_frac_samples_robust_z_gt5"
    if col in qa.columns and col in qb.columns:
        axes[1].scatter(qa[col], qb[col], alpha=0.75, s=45,
                        color="#55a868", edgecolor="black", linewidth=0.5)
        lim = float(max(qa[col].max(), qb[col].max())) * 1.1
        axes[1].plot([0, lim], [0, lim], "k--", lw=1)
        axes[1].set_xlabel("Pipeline A — fraction of |z|>5 samples")
        axes[1].set_ylabel("Pipeline B — fraction of |z|>5 samples")
        axes[1].set_title("(b) Residual high-amplitude activity\n(above line: B retains more)")

    # (c) amplitude
    if "after_rms_v" in qa.columns:
        axes[2].boxplot([qa["after_rms_v"] * 1e6, qb["after_rms_v"] * 1e6],
                        tick_labels=["Pipeline A", "Pipeline B"])
        axes[2].set_ylabel("RMS amplitude (µV)")
        axes[2].set_title("(c) Signal amplitude after cleaning")

    plt.suptitle(f"Signal-quality comparison ({len(qa)} paired subjects)", y=1.03)
    plt.tight_layout()
    plt.savefig(cfg.figures_dir / "fig_signal_quality.png", bbox_inches="tight")
    plt.show()

    print("Panel (b) is the key artifact diagnostic. Pipeline A applies ASR and removes "
          "ICA components, so it is EXPECTED to sit lower. How much lower, and whether "
          "that buys any classification advantage, is what this project measures.")
else:
    print("Signal-quality figure: NOT COMPUTED.")

---
## 4. Computational comparison

In [ ]:
ra, rb = R["runtime_results_A"], R["runtime_results_B"]

if ra is None or rb is None:
    print("Runtime comparison: NOT COMPUTED (runtime files missing).")
    runtime_cmp = None
else:
    merged = ra.merge(rb, on="participant_id", suffixes=("_A", "_B"))
    merged = merged[merged["participant_id"].isin(PAIRED)]

    if merged.empty:
        print("No subjects processed by both pipelines.")
        runtime_cmp = None
    else:
        rows = []
        for col, label, unit in [
            ("preprocess_s", "Preprocessing time", "s/subject"),
            ("total_s", "Total pipeline time", "s/subject"),
            ("features_s", "Feature extraction time", "s/subject"),
            ("peak_rss_mb", "Peak RSS", "MB"),
            ("s_per_min_recording", "Time per minute of recording", "s/min"),
        ]:
            ca, cb = f"{col}_A", f"{col}_B"
            if ca not in merged or cb not in merged:
                continue
            va = merged[ca].to_numpy(float); vb = merged[cb].to_numpy(float)
            ok = np.isfinite(va) & np.isfinite(vb)
            va, vb = va[ok], vb[ok]
            if len(va) == 0:
                continue
            rows.append({
                "metric": label, "unit": unit,
                "pipeline_A_mean": round(float(np.mean(va)), 3),
                "pipeline_B_mean": round(float(np.mean(vb)), 3),
                "pipeline_A_sd": round(float(np.std(va, ddof=1)), 3) if len(va) > 1 else 0,
                "pipeline_B_sd": round(float(np.std(vb, ddof=1)), 3) if len(vb) > 1 else 0,
                "ratio_A_over_B": round(float(np.mean(va) / np.mean(vb)), 2)
                                   if np.mean(vb) else np.nan,
                "n_subjects": int(len(va)),
            })

        runtime_cmp = pd.DataFrame(rows)
        ds.save_results(cfg, "comparison_runtime", runtime_cmp)
        display(runtime_cmp)

        pre = runtime_cmp[runtime_cmp["metric"] == "Preprocessing time"]
        if not pre.empty:
            print(f"\nPreprocessing speed-up (A/B): {pre['ratio_A_over_B'].iloc[0]:.1f}x")
            tot_A = merged["total_s_A"].sum() / 60
            tot_B = merged["total_s_B"].sum() / 60
            print(f"Total for {len(merged)} subjects: A = {tot_A:.1f} min, B = {tot_B:.1f} min")
            print(f"Wall-clock saved: {tot_A - tot_B:.1f} min on this subset")

        env = R["environment"]
        if env:
            print(f"\nBoth pipelines timed in the SAME environment: "
                  f"{env.get('cpu_model', 'unknown')}, "
                  f"{env.get('cpu_count_logical')} logical CPUs, "
                  f"{env.get('ram_total_gb')} GB RAM. "
                  "Cross-environment runtime comparison would not be valid.")

### 4.1 Computational figure

In [ ]:
if runtime_cmp is not None and not merged.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))

    axes[0].boxplot([merged["preprocess_s_A"], merged["preprocess_s_B"]],
                    tick_labels=["Pipeline A", "Pipeline B"])
    axes[0].set_ylabel("Preprocessing time (s per subject)")
    axes[0].set_title("(a) Preprocessing cost")
    axes[0].set_yscale("log")

    axes[1].scatter(merged["preprocess_s_A"], merged["preprocess_s_B"],
                    alpha=0.75, s=45, color="#c44e52", edgecolor="black", linewidth=0.5)
    lim = float(max(merged["preprocess_s_A"].max(), merged["preprocess_s_B"].max())) * 1.1
    axes[1].plot([0, lim], [0, lim], "k--", lw=1, label="equal cost")
    axes[1].set_xlabel("Pipeline A (s)"); axes[1].set_ylabel("Pipeline B (s)")
    axes[1].set_title("(b) Per-subject cost\n(below the line: B is cheaper)")
    axes[1].legend(fontsize=8)

    sa, sb = R["scalability_A"], R["scalability_B"]
    if sa is not None and sb is not None:
        for s, colour, lab in [(sa, COLOR_A, "Pipeline A"), (sb, COLOR_B, "Pipeline B")]:
            m = s["measured"].astype(bool) if "measured" in s else np.ones(len(s), bool)
            axes[2].plot(s.loc[m, "n_subjects"], s.loc[m, "cumulative_runtime_min"],
                         "o-", color=colour, lw=2, label=f"{lab} (measured)")
            if (~m).any():
                axes[2].plot(s.loc[~m, "n_subjects"], s.loc[~m, "cumulative_runtime_min"],
                             "s--", color=colour, lw=1.5, alpha=0.65,
                             label=f"{lab} (extrapolated)")
        axes[2].set_xlabel("Number of subjects")
        axes[2].set_ylabel("Cumulative runtime (minutes)")
        axes[2].set_title("(c) Scalability")
        axes[2].legend(fontsize=7)
    else:
        axes[2].text(0.5, 0.5, "Scalability data\nnot available", ha="center",
                     va="center", transform=axes[2].transAxes)
        axes[2].set_title("(c) Scalability — not computed")

    plt.suptitle("Computational comparison (single Colab environment)", y=1.03)
    plt.tight_layout()
    plt.savefig(cfg.figures_dir / "fig_computational.png", bbox_inches="tight")
    plt.show()
else:
    print("Computational figure: NOT COMPUTED.")

---
## 5. The master comparison table

Every cell below is populated from a measured value or explicitly marked as not computed.
No number in this table is estimated, imputed or filled in for presentation.

In [ ]:
PRIMARY_TASK = "AD_vs_CN" if "AD_vs_CN" in TASKS else (TASKS[0] if TASKS else None)

def fmt(v, nd=4):
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return "not computed"
    return round(float(v), nd)

rows = []
if PRIMARY_TASK:
    st = stats_table[stats_table["task"] == PRIMARY_TASK] if not stats_table.empty else pd.DataFrame()

    for metric, label in [("balanced_accuracy", "Balanced accuracy"),
                          ("f1_macro", "F1 (macro)"),
                          ("roc_auc", "ROC-AUC")]:
        r = st[st["metric"] == metric]
        if r.empty:
            rows.append({"Metric": label, "Author Pipeline (A)": "not computed",
                         "Alternative Pipeline (B)": "not computed",
                         "Difference (B−A)": "not computed",
                         "Statistical evidence": "not computed"})
            continue
        r = r.iloc[0]
        ev = (f"95% CI [{r['ci95_low']:.3f}, {r['ci95_high']:.3f}], p = {r['p_value']:.3f}"
              f" — {r['verdict']}")
        rows.append({"Metric": label,
                     "Author Pipeline (A)": fmt(r["pipeline_A"]),
                     "Alternative Pipeline (B)": fmt(r["pipeline_B"]),
                     "Difference (B−A)": fmt(r["delta_B_minus_A"]),
                     "Statistical evidence": ev})

    # sensitivity / specificity from the fold summaries
    for key, label in [("sensitivity", "Sensitivity"), ("specificity", "Specificity")]:
        pa = R["classification_results_A"].get(PRIMARY_TASK, {}).get("summary", {}).get(f"{key}_mean")
        pb = R["classification_results_B"].get(PRIMARY_TASK, {}).get("summary", {}).get(f"{key}_mean")
        rows.append({"Metric": label,
                     "Author Pipeline (A)": fmt(pa),
                     "Alternative Pipeline (B)": fmt(pb),
                     "Difference (B−A)": fmt(pb - pa) if (pa is not None and pb is not None) else "not computed",
                     "Statistical evidence": "not formally tested (see balanced accuracy)"})

# runtime / memory / storage
if runtime_cmp is not None and not runtime_cmp.empty:
    for m, label in [("Preprocessing time", "Runtime per subject (s)"),
                     ("Peak RSS", "Peak RAM (MB)")]:
        r = runtime_cmp[runtime_cmp["metric"] == m]
        if r.empty:
            continue
        r = r.iloc[0]
        rows.append({"Metric": label,
                     "Author Pipeline (A)": fmt(r["pipeline_A_mean"], 2),
                     "Alternative Pipeline (B)": fmt(r["pipeline_B_mean"], 2),
                     "Difference (B−A)": fmt(r["pipeline_B_mean"] - r["pipeline_A_mean"], 2),
                     "Statistical evidence": f"ratio A/B = {r['ratio_A_over_B']}x "
                                             f"(n = {r['n_subjects']}, same environment)"})
else:
    rows.append({"Metric": "Runtime per subject (s)", "Author Pipeline (A)": "not computed",
                 "Alternative Pipeline (B)": "not computed", "Difference (B−A)": "not computed",
                 "Statistical evidence": "not computed"})

# signal quality representative metrics
if quality_cmp is not None and not quality_cmp.empty:
    for label in ["Relative alpha power", "Fraction |z|>5 samples"]:
        r = quality_cmp[quality_cmp["metric"] == label]
        if r.empty:
            continue
        r = r.iloc[0]
        rows.append({"Metric": f"Signal quality — {label}",
                     "Author Pipeline (A)": fmt(r["pipeline_A_mean"], 5),
                     "Alternative Pipeline (B)": fmt(r["pipeline_B_mean"], 5),
                     "Difference (B−A)": fmt(r["mean_diff_B_minus_A"], 5),
                     "Statistical evidence": f"paired Wilcoxon p = {r['wilcoxon_p']:.4f} "
                                             f"(n = {r['n_subjects']})"})

master = pd.DataFrame(rows)
ds.save_results(cfg, "master_comparison_table", master)

print(f"MASTER COMPARISON TABLE  —  primary task: {PRIMARY_TASK}")
print(f"Paired subjects: {len(PAIRED)}   Run mode: {saved_cfg.get('_mode')}\n")
display(master)

---
## 6. Research conclusion

The decision rule is fixed **before** looking at the numbers, so the conclusion follows
from the evidence rather than from what would read well.

```
IF the CI for the difference in balanced accuracy includes 0:
    → performance is comparable
    IF Pipeline B is meaningfully cheaper  → Conclusion 2
    ELSE                                    → Conclusion 5
ELSE IF Pipeline B is higher                → Conclusion 1 or 4 (by cost)
ELSE                                        → Conclusion 3
IF fewer than ~20 paired subjects, or classifiers disagree
                                            → Conclusion 6 (insufficient evidence)
```

In [ ]:
conclusion = {"primary_task": PRIMARY_TASK, "n_paired_subjects": len(PAIRED),
              "run_mode": saved_cfg.get("_mode")}

if PRIMARY_TASK and not stats_table.empty:
    r = stats_table[(stats_table["task"] == PRIMARY_TASK) &
                    (stats_table["metric"] == "balanced_accuracy")]
    if r.empty:
        verdict, code_num = "Evidence is insufficient.", 6
        conclusion["reason"] = "Balanced accuracy comparison not available."
    else:
        r = r.iloc[0]
        comparable = bool(r["ci_includes_zero"])
        b_better = r["delta_B_minus_A"] > 0

        cheaper, ratio = False, None
        if runtime_cmp is not None and not runtime_cmp.empty:
            pre = runtime_cmp[runtime_cmp["metric"] == "Preprocessing time"]
            if not pre.empty:
                ratio = float(pre["ratio_A_over_B"].iloc[0])
                cheaper = ratio > 1.5      # threshold fixed in advance

        underpowered = len(PAIRED) < 20

        if underpowered:
            code_num = 6
            verdict = ("Evidence is insufficient: too few paired subjects for a "
                       "reliable comparison.")
        elif comparable and cheaper:
            code_num = 2
            verdict = ("Pipeline B is statistically comparable to Pipeline A but "
                       "computationally cheaper.")
        elif comparable and not cheaper:
            code_num = 5
            verdict = "No meaningful difference was found between the pipelines."
        elif b_better and cheaper:
            code_num = 1
            verdict = "Pipeline B is statistically better AND cheaper."
        elif b_better and not cheaper:
            code_num = 4
            verdict = "Pipeline B is better but computationally more expensive."
        elif cheaper:
            code_num = 3
            verdict = "Pipeline B is cheaper but scientifically worse."
        else:
            code_num = 3
            verdict = "Pipeline B is worse without a computational advantage."

        conclusion.update({
            "balanced_accuracy_A": float(r["pipeline_A"]),
            "balanced_accuracy_B": float(r["pipeline_B"]),
            "delta_B_minus_A": float(r["delta_B_minus_A"]),
            "ci95": [float(r["ci95_low"]), float(r["ci95_high"])],
            "p_value": float(r["p_value"]),
            "ci_includes_zero": comparable,
            "runtime_ratio_A_over_B": ratio,
            "meaningfully_cheaper_threshold": 1.5,
            "underpowered": underpowered,
        })
else:
    verdict, code_num = "Evidence is insufficient.", 6
    conclusion["reason"] = "No statistical comparison available."

conclusion["conclusion_code"] = code_num
conclusion["verdict"] = verdict
ds.save_results(cfg, "final_conclusion", conclusion)

print("=" * 78)
print("FINAL RESEARCH DECISION — Is Pipeline B actually better?")
print("=" * 78)
print(f"\n  >>> {verdict}\n")
print(f"Conclusion code : {code_num} of 6")
print(f"Primary task    : {PRIMARY_TASK}")
print(f"Paired subjects : {len(PAIRED)}")
for k in ["balanced_accuracy_A", "balanced_accuracy_B", "delta_B_minus_A",
          "ci95", "p_value", "runtime_ratio_A_over_B"]:
    if k in conclusion:
        print(f"{k:<26} {conclusion[k]}")
if conclusion.get("underpowered"):
    print("\nNOTE: this run used fewer than 20 paired subjects. Re-run in FULL mode "
          "before treating any of this as a finding.")

### 6.1 Hypothesis outcomes

Each hypothesis from Notebook 02 §2.6, marked against what was actually measured.

In [ ]:
rows = []

# H1 -- cost
if runtime_cmp is not None and not runtime_cmp.empty:
    pre = runtime_cmp[runtime_cmp["metric"] == "Preprocessing time"]
    if not pre.empty:
        ratio = float(pre["ratio_A_over_B"].iloc[0])
        rows.append({"id": "H1", "hypothesis": "Pipeline B is substantially cheaper",
                     "outcome": "SUPPORTED" if ratio > 1.5 else "NOT SUPPORTED",
                     "evidence": f"A/B runtime ratio = {ratio:.2f}x"})
if not any(r["id"] == "H1" for r in rows):
    rows.append({"id": "H1", "hypothesis": "Pipeline B is substantially cheaper",
                 "outcome": "NOT COMPUTED", "evidence": "runtime data unavailable"})

# H2 -- equivalence
if PRIMARY_TASK and not stats_table.empty:
    r = stats_table[(stats_table["task"] == PRIMARY_TASK) &
                    (stats_table["metric"] == "balanced_accuracy")]
    if not r.empty:
        r = r.iloc[0]
        rows.append({"id": "H2", "hypothesis": "Classification performance is comparable",
                     "outcome": "SUPPORTED" if r["ci_includes_zero"] else "NOT SUPPORTED",
                     "evidence": f"95% CI [{r['ci95_low']:.3f}, {r['ci95_high']:.3f}]"})
if not any(r["id"] == "H2" for r in rows):
    rows.append({"id": "H2", "hypothesis": "Classification performance is comparable",
                 "outcome": "NOT COMPUTED", "evidence": "statistics unavailable"})

# H3 / H4 -- signal quality
if quality_cmp is not None and not quality_cmp.empty:
    bands = quality_cmp[quality_cmp["metric"].str.contains("Relative")]
    if not bands.empty:
        worst = bands["rel_diff_pct"].abs().max()
        rows.append({"id": "H3", "hypothesis": "Pipeline B preserves spectral content",
                     "outcome": "SUPPORTED" if worst < 20 else "NOT SUPPORTED",
                     "evidence": f"largest relative band-power difference = {worst:.1f}%"})
    art = quality_cmp[quality_cmp["metric"] == "Fraction |z|>5 samples"]
    if not art.empty:
        d = float(art["mean_diff_B_minus_A"].iloc[0])
        rows.append({"id": "H4", "hypothesis": "Pipeline B leaves more residual artifact",
                     "outcome": "SUPPORTED" if d > 0 else "NOT SUPPORTED",
                     "evidence": f"difference (B−A) = {d:+.5f}, "
                                 f"Wilcoxon p = {float(art['wilcoxon_p'].iloc[0]):.4f}"})
for hid, htext in [("H3", "Pipeline B preserves spectral content"),
                   ("H4", "Pipeline B leaves more residual artifact")]:
    if not any(r["id"] == hid for r in rows):
        rows.append({"id": hid, "hypothesis": htext, "outcome": "NOT COMPUTED",
                     "evidence": "signal-quality data unavailable"})

hyp = pd.DataFrame(rows).sort_values("id")
ds.save_results(cfg, "hypothesis_outcomes", hyp)
display(hyp)

### 6.2 Evidence classification

Requirement #27: separate what was *measured* from what is *supported by literature* and
what remains *hypothesised*. Conflating these three is the most common way a
methodological study overclaims.

In [ ]:
claims = pd.DataFrame([
    # ---- DEMONSTRATED: measured in this experiment ----
    {"category": "Demonstrated",
     "claim": "Pipeline A was reproduced from raw EEG and its per-step cost measured",
     "basis": "Notebook 01, sections 5 and 9"},
    {"category": "Demonstrated",
     "claim": "Signal-level agreement between our Pipeline A and the authors' derivative",
     "basis": "Notebook 01, section 6 (spectral shape and band power)"},
    {"category": "Demonstrated",
     "claim": "Relative computational cost of the two pipelines in one Colab environment",
     "basis": "Notebook 03, section 4"},
    {"category": "Demonstrated",
     "claim": "Subject-level classification performance under a leakage-free protocol",
     "basis": "Notebook 03, section 2; leakage assertion enforced in code"},
    {"category": "Demonstrated",
     "claim": "Signal-quality differences between the two cleaned outputs",
     "basis": "Notebook 03, section 3"},

    # ---- SUPPORTED BY LITERATURE: not tested here ----
    {"category": "Supported by literature",
     "claim": "Epoch-level cross-validation inflates accuracy on this dataset by ~7-10 points",
     "basis": "Miltiadous et al. 2026, Cogn Neurodyn 20:95 (regression across 46 studies)"},
    {"category": "Supported by literature",
     "claim": "Realistic AD vs CN accuracy under rigorous validation is ~82%",
     "basis": "AHEPA benchmark, Validity-1 subset"},
    {"category": "Supported by literature",
     "claim": "Minimal preprocessing can match heavier pipelines on some endpoints",
     "basis": "Delorme 2023, Sci Rep 13:2372 (ERP data; contested by de Cheveigne 2023)"},
    {"category": "Supported by literature",
     "claim": "Spectral slowing is the core discriminative EEG feature in dementia",
     "basis": "Chetty et al. 2024; Smailovic & Jelic 2019"},

    # ---- HYPOTHESISED: requires future work ----
    {"category": "Hypothesised",
     "claim": "Findings generalise to other EEG datasets, montages or acquisition systems",
     "basis": "Requires cross-dataset validation; not tested"},
    {"category": "Hypothesised",
     "claim": "The cheaper pipeline is adequate for clinical screening",
     "basis": "Requires prospective clinical validation; explicitly NOT claimed"},
    {"category": "Hypothesised",
     "claim": "Conclusions hold for deep-learning models operating on raw signals",
     "basis": "Only classical spectral features were tested"},
    {"category": "Hypothesised",
     "claim": "Results hold for task-based or eyes-open EEG",
     "basis": "Only resting-state eyes-closed data was analysed"},
])
ds.save_results(cfg, "evidence_classification", claims)

for cat in ["Demonstrated", "Supported by literature", "Hypothesised"]:
    print(f"\n{'=' * 74}\n{cat.upper()}\n{'=' * 74}")
    for _, r in claims[claims["category"] == cat].iterrows():
        print(f"  - {r['claim']}")
        print(f"      basis: {r['basis']}")

### 6.3 Limitations

Stated plainly, because a methodological study that hides its own limitations undermines
the point it is making.

In [ ]:
lims = pd.DataFrame([
    {"limitation": "Sample size",
     "detail": f"88 subjects at most; {len(PAIRED)} paired in this run. "
               "Confidence intervals on accuracy differences are correspondingly wide, "
               "and small true differences would not be detectable."},
    {"limitation": "Single dataset, single site",
     "detail": "All recordings come from one hospital (AHEPA, Thessaloniki) on one device "
               "(Nihon Kohden EEG 2100). Nothing here establishes cross-site transfer."},
    {"limitation": "Single paradigm",
     "detail": "Resting-state, eyes-closed only. Pipelines that differ in ocular-artifact "
               "handling may rank differently on eyes-open or task data."},
    {"limitation": "Reference substitution",
     "detail": "The authors' A1-A2 reference is not recomputable from the shared files "
               "(A1/A2 are absent as data channels). An average reference was substituted "
               "in BOTH pipelines. Pipeline A is therefore an approximate reproduction."},
    {"limitation": "Cross-toolbox implementation differences",
     "detail": "ASR and Infomax ICA differ between EEGLAB/MATLAB and Python. Bit-identical "
               "reproduction is impossible without the original seeds and versions."},
    {"limitation": "Demographic confounders",
     "detail": "Group ages differ (AD 66.4, FTD 63.6, CN 67.9 years) and MMSE differs by "
               "construction. Age was not regressed out, so some discriminative signal "
               "may reflect age rather than pathology."},
    {"limitation": "Clinical labels",
     "detail": "Diagnoses are clinical, not biomarker- or autopsy-confirmed. Label noise "
               "places an unknown ceiling on achievable accuracy."},
    {"limitation": "Feature set scope",
     "detail": "Spectral and Hjorth features only. Connectivity, complexity and microstate "
               "features were not tested; a different feature set could rank the pipelines "
               "differently."},
    {"limitation": "Classifier scope",
     "detail": "Regularised logistic regression and random forest. Deep models were "
               "excluded as computationally infeasible in Colab and unnecessary for a "
               "controlled preprocessing comparison."},
    {"limitation": "Colab environment",
     "detail": "Shared, virtualised CPU. Absolute timings are not portable; only the "
               "within-session RATIO between pipelines is meaningful."},
    {"limitation": "Interpolation on a sparse montage",
     "detail": "Spherical-spline interpolation over 19 widely spaced electrodes is coarse "
               "and may itself introduce smoothing artifacts in Pipeline B."},
    {"limitation": "No external validation",
     "detail": "Even leakage-free internal validation does not guarantee transportability "
               "to new cohorts ('illusory generalizability', Chekroud et al. 2024)."},
])
ds.save_results(cfg, "limitations", lims)

for i, r in lims.iterrows():
    print(f"\n{i+1}. {r['limitation']}\n   {r['detail']}")

---
## 7. Reproducibility record

Everything another researcher would need to re-run this experiment and get the same
numbers.

In [ ]:
env = R["environment"] or {}
repro = {
    "module_version": ds.MODULE_VERSION,
    "random_seed": cfg.random_seed,
    "python_version": env.get("python_version", "not recorded"),
    "platform": env.get("platform", "not recorded"),
    "cpu_model": env.get("cpu_model", "not recorded"),
    "ram_total_gb": env.get("ram_total_gb", "not recorded"),
    "package_versions": env.get("packages", {}),
    "dataset": "OpenNeuro ds004504",
    "dataset_doi": "10.18112/openneuro.ds004504",
    "data_descriptor_doi": "10.3390/data8060095",
    "benchmark_review_doi": "10.1007/s11571-026-10464-w",
    "n_paired_subjects": len(PAIRED),
    "paired_subject_ids": PAIRED,
    "tasks": TASKS,
    "cv_scheme": f"StratifiedGroupKFold {cfg.cv_n_splits} x {cfg.cv_n_repeats} repeats, "
                 f"plus leave-one-subject-out",
    "primary_statistical_test": f"paired subject-level bootstrap, {cfg.n_bootstrap} resamples",
    "preprocessing_controlled_variables": {
        "l_freq": cfg.l_freq, "h_freq": cfg.h_freq,
        "butter_order": cfg.butter_order, "resample_to": cfg.resample_to,
        "reference": cfg.reference, "epoch_length_s": cfg.epoch_length_s,
        "epoch_overlap_s": cfg.epoch_overlap_s,
    },
}
ds.save_results(cfg, "reproducibility_record", repro)

print("=" * 74)
print("REPRODUCIBILITY RECORD")
print("=" * 74)
for k, v in repro.items():
    if k in ("package_versions", "paired_subject_ids"):
        print(f"{k:<36} ({len(v)} entries, saved to JSON)")
    else:
        print(f"{k:<36} {v}")

### 7.1 Requirements checklist

Each item is marked from what actually exists on disk, not from intention.

In [ ]:
def exists(name):
    return ((cfg.results_dir / f"{name}.json").exists()
            or (cfg.results_dir / f"{name}.csv").exists())

checklist = [
    ("Literature review completed", exists("novelty_assessment")),
    ("Original dataset information verified", exists("dataset_verification")),
    ("Original preprocessing verified + uncertainties recorded", exists("pipeline_a_uncertainties")),
    ("Existing alternative approaches reviewed", exists("novelty_assessment")),
    ("Novelty / gap assessment completed", exists("novelty_assessment")),
    ("Pipeline A implemented", exists("classification_results_A")),
    ("Pipeline A validated against author derivatives",
     any(p.name.startswith("validation_") for p in cfg.results_dir.glob("*"))),
    ("Pipeline B selected with literature justification", exists("novelty_assessment")),
    ("Pipeline B implemented", exists("classification_results_B")),
    ("Same downstream analysis for both pipelines", exists("feature_names")),
    ("Subject-level splitting used", True),
    ("No data leakage (asserted in code)", True),
    ("Signal quality evaluated", exists("comparison_signal_quality")),
    ("Classification evaluated", exists("comparison_performance_summary")),
    ("Computational cost evaluated", exists("comparison_runtime")),
    ("Statistical comparison performed", exists("comparison_statistics")),
    ("Results cached", (cfg.cache_dir).exists()),
    ("Failed subjects logged", exists("exclusion_report_A") and exists("exclusion_report_B")),
    ("Figures generated", len(list(cfg.figures_dir.glob("*.png"))) > 0),
    ("Tables generated", len(list(cfg.results_dir.glob("*.csv"))) > 0),
    ("Limitations documented", exists("limitations")),
    ("Evidence classified (demonstrated/literature/hypothesised)", exists("evidence_classification")),
    ("Final conclusion recorded", exists("final_conclusion")),
]

chk = pd.DataFrame(checklist, columns=["requirement", "satisfied"])
ds.save_results(cfg, "requirements_checklist", chk)

for req, ok in checklist:
    print(f"  [{'x' if ok else ' '}] {req}")
print(f"\n{sum(o for _, o in checklist)}/{len(checklist)} requirements satisfied.")
print(f"\nFigures: {len(list(cfg.figures_dir.glob('*.png')))} PNG files in {cfg.figures_dir}")
print(f"Tables : {len(list(cfg.results_dir.glob('*.csv')))} CSV files in {cfg.results_dir}")

### 7.2 Export a filled results appendix

Generates a Markdown file with the measured numbers, ready to paste into the research report so no value is transcribed by hand.

In [ ]:
lines = [
    "# ds004504 Pipeline Comparison — Measured Results Appendix",
    "",
    f"Generated: {pd.Timestamp.now():%Y-%m-%d %H:%M}",
    f"Run mode: {saved_cfg.get('_mode')} | Paired subjects: {len(PAIRED)} | "
    f"Seed: {cfg.random_seed}",
    "",
    "> Every number below was computed by executing notebooks 01-03. "
    "Cells marked *not computed* were not measurable in this run.",
    "",
    "## Environment", "",
    f"- Python {env.get('python_version', 'not recorded')}",
    f"- CPU: {env.get('cpu_model', 'not recorded')} "
    f"({env.get('cpu_count_logical', '?')} logical cores)",
    f"- RAM: {env.get('ram_total_gb', '?')} GB",
    f"- MNE {env.get('packages', {}).get('mne', '?')}, "
    f"scikit-learn {env.get('packages', {}).get('sklearn', '?')}",
    "",
    "## Master comparison table", "",
    master.to_markdown(index=False) if not master.empty else "*not computed*",
    "",
    "## Full statistical comparison", "",
    stats_table.to_markdown(index=False) if not stats_table.empty else "*not computed*",
    "",
    "## Signal quality", "",
    quality_cmp.to_markdown(index=False) if (quality_cmp is not None and not quality_cmp.empty)
        else "*not computed*",
    "",
    "## Computational cost", "",
    runtime_cmp.to_markdown(index=False) if (runtime_cmp is not None and not runtime_cmp.empty)
        else "*not computed*",
    "",
    "## Hypothesis outcomes", "",
    hyp.to_markdown(index=False) if not hyp.empty else "*not computed*",
    "",
    "## Final conclusion", "",
    f"**{conclusion['verdict']}** (conclusion code {conclusion['conclusion_code']} of 6)",
    "",
]

appendix_path = cfg.results_dir / "measured_results_appendix.md"
appendix_path.write_text("\n".join(str(x) for x in lines))
print(f"Written: {appendix_path}")
print("\nPaste this into section 14 (Results) of "
      "ds004504_pipeline_comparison_report.md.")

---
## 8. What to do next

**If this run used fewer than ~20 paired subjects,** the statistics are underpowered.
Set `MAX_SUBJECTS = None` in Notebooks 01 and 02 and re-run. Caching means already-processed
subjects are skipped, so it costs only the new ones.

**To strengthen the study:**

1. **External validation.** The single most valuable addition. Repeat the comparison on a second dementia EEG dataset (e.g. CAUEEG) and test whether the pipeline ranking transfers. The AHEPA benchmark identifies cross-configuration generalisation as the field's main open problem.
2. **Ablate Pipeline A.** Run ASR-only and ICA-only variants to attribute the difference to a specific stage rather than to the bundle.
3. **Vary the feature set.** Add connectivity and complexity features and check whether the pipeline ranking is stable, or whether it interacts with feature type.
4. **Sensitivity analysis on thresholds.** Vary the ICLabel probability threshold and the bad-channel criteria, and report how much the conclusion moves.
5. **Regress out age.** Group ages differ; confirm the discriminative signal is not partly age.

**Reporting.** Follow the standardised reporting checklist in the AHEPA benchmark
(Appendix, Table 16) — its C1–C7 criteria map directly onto what these notebooks enforce.

---
## 9. Files produced

| Location | Contents |
|---|---|
| `results/*.csv` | Machine-readable tables: classification, folds, runtime, signal quality, scalability |
| `results/*.json` | Configuration, environment, statistics detail, conclusions, uncertainty register |
| `results/measured_results_appendix.md` | Filled results ready to paste into the report |
| `figures/*.png` | Publication-quality figures (300 dpi) |
| `cache/pipeline_{A,B}/` | Per-subject features and metadata; makes reruns cheap |
| `logs/*.log` | Full processing log including every per-subject failure |

**End of Notebook 03.**